# Week 6 - Text Representation II : Distributed Representation (1/2)
**Notebook 1 : Word-level Embeddings - Word2Vec, GloVe, FastText**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

In Week 4 every word was an independent column of a count matrix, so `good` and `great` were as unrelated as
`good` and `banana`. Lecture 5-1 / 5-2 *Text Representation II - Distributed Representation* replaces that column
with a short **dense vector** learned from the contexts in which the word appears.

This notebook follows chapters 1 to 3 of the lecture:

| Lecture chapter | What we do in code |
|---|---|
| Word embedding: one-hot vs distributed | show that one-hot vectors of `motel` and `hotel` have a dot product of 0 |
| 1. Word2Vec | build the skip-gram training samples by hand, compute the subsampling probability, train Skip-gram and CBOW, look at the **two vectors** of every word |
| 2. GloVe | build a co-occurrence matrix, compute the probability ratio $P_{ik}/P_{jk}$, plot the weighting function $f(x)$, use pre-trained vectors |
| 3. FastText | list the character n-grams of a word, give a vector to a word that was never seen |
| (evaluation) | word similarity test and word analogy test |

Chapter 4 (sentence / paragraph / document level) is Notebook 2.

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom (`Shift + Enter`).

## &nbsp;0. Setup
[Gensim](https://radimrehurek.com/gensim/intro.html) provides Word2Vec, FastText and a downloader for pre-trained vectors.

> ⚠️ If Colab asks you to **restart the session** after the install, do it (`Runtime ▸ Restart session`)
> and run the cells again from here. It happens when gensim needs a different NumPy version than the one already loaded.

In [ ]:
!pip install -q gensim

import nltk
for r in ['movie_reviews', 'punkt', 'punkt_tab', 'stopwords']:
    nltk.download(r, quiet=True)

In [ ]:
import gensim
from gensim.models import Word2Vec, FastText, KeyedVectors

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

print('gensim version:', gensim.__version__)

## &nbsp;1. One-hot vs Distributed Representation
The lecture starts from the simplest word vector: a **one-hot vector** of the size of the vocabulary.
It is a valid vector, but *"similarities between words cannot be preserved"*:
the dot product of any two different one-hot vectors is 0.

In [ ]:
vocab = ['aardvark', 'a', 'at', 'cat', 'hotel', 'motel', 'zebra']

def one_hot(word):
    v = np.zeros(len(vocab), dtype=int)
    v[vocab.index(word)] = 1
    return v

print('motel :', one_hot('motel'))
print('hotel :', one_hot('hotel'))
print()
print('hotel . motel =', one_hot('hotel') @ one_hot('motel'))
print('hotel . cat   =', one_hot('hotel') @ one_hot('cat'))

`motel` is as far from `hotel` as `cat` is. A **distributed representation** is a parameterized function

$$W : \text{words} \rightarrow \mathbb{R}^n, \qquad W(\text{"cat"}) = (0.2, -0.4, 0.7, \dots)$$

with a small $n$ (50 - 300), whose values are *learned* so that semantically similar words are close to each other.
Everything below is a different way of learning $W$.

## &nbsp;2. Word2Vec
Two architectures (lecture: *CBOW vs Skip-gram*):

* **Skip-gram** : predict the surrounding words in a window of length $m$ from the center word, $p(w_{t+j} \mid w_t)$
* **CBOW** : predict the center word from the sum of its context words, $p(w \mid \mathcal{C})$

### &nbsp;2-1. Training samples by hand
*Learning strategy* slide: with window = 2, the sentence *"The quick brown fox jumps over the lazy dog."*
produces one (center, context) pair for every neighbor within 2 positions. The network never sees the sentence,
only these pairs.

In [ ]:
def skipgram_pairs(tokens, m=2):
    # (center, context) for every context word within m positions of the center
    pairs = []
    for i, center in enumerate(tokens):
        for j in range(max(0, i - m), min(len(tokens), i + m + 1)):
            if j != i:
                pairs.append((center, tokens[j]))
    return pairs


sentence = 'the quick brown fox jumps over the lazy dog'.split()
pairs = skipgram_pairs(sentence, m=2)

start = 0
for i in range(4):                                   # the first four positions, as on the slide
    k = min(i, 2) + min(len(sentence) - 1 - i, 2)    # how many neighbors position i has
    print(f'{sentence[i]:<6}', pairs[start:start + k])
    start += k
print('\n#pairs from one 9-word sentence:', len(pairs))

The four lines are the four rows of the lecture slide. The first `the` has only two pairs, because nothing is on its left,
and a 9-word sentence already gives 30 training samples.

### &nbsp;2-2. Subsampling frequent words
The number of weights is $2 \times V \times N$, a huge network. Two tricks reduce the work; one of them is
**subsampling**: every occurrence of word $w_i$ is removed from the training text with probability

$$P(w_i) = 1 - \sqrt{\frac{t}{f(w_i)}}$$

where $f(w_i)$ is the word's relative frequency in the corpus and $t$ a threshold ($10^{-5}$ on the slide).

In [ ]:
def discard_prob(f, t=1e-5):
    return max(0.0, 1 - np.sqrt(t / f))     # a word rarer than t is never removed

# the two examples of the slide
print(f'f = 1e-4 -> P = {discard_prob(1e-4):.4f}')
print(f'f = 1e-2 -> P = {discard_prob(1e-2):.4f}')

Now on a real corpus. We use NLTK `movie_reviews` (2,000 reviews), already split into sentences.

In [ ]:
from nltk.corpus import movie_reviews

sentences = [[w.lower() for w in sent if w.isalpha()] for sent in movie_reviews.sents()]
sentences = [s for s in sentences if len(s) > 1]

counts = Counter(w for s in sentences for w in s)
total = sum(counts.values())
print('#sentences:', len(sentences), ' #tokens:', total, ' #distinct words:', len(counts))
print('example   :', sentences[0][:15])

In [ ]:
print(f"{'word':<10}{'count':>8}{'f(w)':>11}{'P, t=1e-5':>12}{'P, t=1e-3':>12}")
for w in ['the', 'film', 'good', 'actor', 'boring', 'spielberg']:
    f = counts[w] / total
    print(f'{w:<10}{counts[w]:>8}{f:>11.2e}{discard_prob(f, 1e-5):>12.3f}{discard_prob(f, 1e-3):>12.3f}')

**🤔 Quick check**
With the slide's $t = 10^{-5}$, even `spielberg` loses most of its occurrences. That threshold was chosen for corpora of
*billions* of tokens; our corpus has 1.3 million, so gensim's default `sample=1e-3` is far more reasonable here.
Notice also what happens to `the`: subsampling does the job of a stop-word list *softly*, without a list.

### &nbsp;2-3. Training Skip-gram with gensim

| parameter | meaning |
|---|---|
| `sentences` | list of token lists |
| `vector_size` | $N$, the dimension of the embedding (default 100) |
| `window` | $m$, how many words to the left / right count as context |
| `min_count` | ignore words that occur fewer than *n* times |
| `sg` | `1` = Skip-gram, `0` = CBOW |
| `negative` | number of negative samples (gensim's default training method, 5) |
| `sample` | subsampling threshold $t$ (default `1e-3`) |
| `epochs` | training passes over the corpus |

In [ ]:
# Training takes about 30 seconds.
w2v_model = Word2Vec(sentences=sentences,
                     vector_size=100,
                     window=5,
                     min_count=5,
                     sg=1,          # skip-gram
                     workers=4,
                     epochs=5,
                     seed=0)

word2vec_vectors = w2v_model.wv          # KeyedVectors: the word -> vector lookup table
print('#vocabulary size:', len(word2vec_vectors))
print('#vector size    :', word2vec_vectors.vector_size)

vec = word2vec_vectors['actor']
print('\nactor ->', vec.shape, vec[:6].round(3), '...')

> `workers=4` uses four threads, so two runs can give slightly different numbers. That is normal;
> `workers=1` together with `seed` makes a run exactly reproducible (slower).

### &nbsp;2-4. Reading the embedding space

In [ ]:
print("most similar to 'actor':")
for word, score in word2vec_vectors.most_similar('actor', topn=8):
    print(f'   {word:<15} {score:.3f}')

In [ ]:
# cosine similarity of two word vectors
for a, b in [('good', 'great'), ('good', 'bad'), ('good', 'actor')]:
    print(f"similarity('{a}', '{b}') = {word2vec_vectors.similarity(a, b):.3f}")

# the odd one out
print("\ndoesnt_match(['comedy', 'drama', 'thriller', 'camera']) ->",
      word2vec_vectors.doesnt_match(['comedy', 'drama', 'thriller', 'camera']))

**🤔 Quick check** `good` and `bad` are close. They are opposites in meaning, but they appear in exactly the same
contexts (*"a ___ movie"*), and context is all that Word2Vec sees.

In [ ]:
# Vector arithmetic (lecture: Male-Female, Verb tense, Country-Capital)
# actor - man + woman = ?
print(word2vec_vectors.most_similar(positive=['actor', 'woman'], negative=['man'], topn=5))

In [ ]:
# Out Of Vocabulary (OOV): a word that never appeared in the corpus has no vector at all.
try:
    word2vec_vectors['hanyang']
except KeyError as e:
    print('KeyError ->', e)

### &nbsp;2-5. Every word has two vectors
The lecture writes the skip-gram probability as

$$p(o \mid c) = \frac{\exp(u_o^\top v_c)}{\sum_{w=1}^{W} \exp(u_w^\top v_c)}$$

* $v$ : the **center** vector, a row of the input matrix $W$ (this is what we call "the word vector")
* $u$ : the **outside** (context) vector, a column of the output matrix $W'$

gensim keeps both. `wv.vectors` is $W$, and with negative sampling the output matrix is stored in `syn1neg`.

> **Note.** gensim does not train this full softmax. With negative sampling (its default) each $u_o^\top v_c$ is only pushed
> up for real context pairs and down for a few random words. Applying the softmax over the whole vocabulary to those
> scores, as below, is an **illustration** of the lecture's formula, not the probability the model was trained to output.

In [ ]:
W_in = w2v_model.wv.vectors        # v : center vectors  (V x N)
W_out = w2v_model.syn1neg          # u : outside vectors (V x N), i.e. W' transposed

V, N = W_in.shape
print('W  (input)  :', W_in.shape)
print("W' (output) :", W_out.shape)
print(f'#weights 2 x V x N = 2 x {V} x {N} = {2 * V * N:,}')

In [ ]:
def context_distribution(center, model):
    v_c = model.wv[center]                 # center vector
    scores = model.syn1neg @ v_c           # u_w . v_c for every word w
    p = np.exp(scores - scores.max())      # subtract the max for numerical stability
    return p / p.sum()


p = context_distribution('actor', w2v_model)
top = np.argsort(-p)[:8]
print('sum of p(o | actor):', round(p.sum(), 4))
print('most probable CONTEXT words :', [w2v_model.wv.index_to_key[i] for i in top])
print('most SIMILAR words (v . v)  :', [w for w, _ in word2vec_vectors.most_similar('actor', topn=8)])

**🤔 Quick check** The two lists answer different questions.
$p(o \mid c)$ uses $u$ and lists the words that appear **next to** `actor` (*best* actor, *award*-winning, *talented*);
`most_similar` compares $v$ with $v$ and lists the words that appear **in the same places as** `actor`
(`actress`, `performer`). Word similarity is the second one, which is why the input matrix is the one we keep.

### &nbsp;2-6. CBOW vs Skip-gram
Same data and parameters, only `sg` changes.

In [ ]:
cbow_model = Word2Vec(sentences=sentences, vector_size=100, window=5,
                      min_count=5, sg=0, workers=4, epochs=5, seed=0)
cbow_vectors = cbow_model.wv

for w in ['actor', 'boring', 'spielberg']:
    print(f'--- {w} ---')
    print('  skip-gram:', [x[0] for x in word2vec_vectors.most_similar(w, topn=5)])
    print('  CBOW     :', [x[0] for x in cbow_vectors.most_similar(w, topn=5)])

CBOW combines the context into one vector before predicting (the lecture writes the **sum** $h_C = \sum_c x_c$; gensim takes the
**mean** by default, `cbow_mean=1`, which only rescales it), so each center word is one training example; Skip-gram
makes one example per (center, context) pair. Skip-gram is slower but tends to do better for rare words such as
proper nouns, CBOW is faster and smooths over frequent words.

## &nbsp;3. GloVe
*Limitation of Word2Vec*: the network spends a lot of time on overwhelmingly frequent words, and it only ever sees
one local window at a time. GloVe (Global Vectors) starts instead from **global statistics**: the word-word
co-occurrence matrix of the whole corpus.

### &nbsp;3-1. The co-occurrence matrix
Notation of the lecture:

* $X_{ij}$ : how often word $j$ occurs in the context of word $i$
* $X_i = \sum_k X_{ik}$ : the total number of context words of $i$ (**the row sum**, not the frequency of $i$)
* $P_{ij} = P(j \mid i) = X_{ij} / X_i$

In [ ]:
toy = ['i like deep learning',
       'i like nlp',
       'i enjoy flying']
toy = [s.split() for s in toy]

def cooccurrence(sents, window=1):
    X = defaultdict(Counter)
    for s in sents:
        for i, w in enumerate(s):
            for j in range(max(0, i - window), min(len(s), i + window + 1)):
                if j != i:
                    X[w][s[j]] += 1
    return X

X_toy = cooccurrence(toy, window=1)
words = ['i', 'like', 'enjoy', 'deep', 'learning', 'nlp', 'flying']
pd.DataFrame([[X_toy[a][b] for b in words] for a in words], index=words, columns=words)

In [ ]:
X_i = sum(X_toy['like'].values())
print('X_like =', X_i)
print('P(i | like)    =', X_toy['like']['i'] / X_i)
print('P(deep | like) =', X_toy['like']['deep'] / X_i)

### &nbsp;3-2. The ratio of co-occurrence probabilities
*Motivation* slide: on its own $P(k \mid ice)$ says little, but the **ratio** $P(k \mid ice) / P(k \mid steam)$
separates the probe words $k$:

| | $k$ = solid | $k$ = gas | $k$ = water | $k$ = fashion |
|---|---|---|---|---|
| ratio | 8.9 (large) | 0.085 (small) | 1.36 (≈1) | 0.96 (≈1) |

`ice` and `steam` are too rare in movie reviews, so we repeat the idea with `actor` vs `director`, window 5.

In [ ]:
X = cooccurrence(sentences, window=5)

def ratio_table(i, j, probes):
    Xi, Xj = sum(X[i].values()), sum(X[j].values())
    rows = []
    for k in probes:
        p_i, p_j = X[i][k] / Xi, X[j][k] / Xj
        rows.append({'k': k, f'P(k|{i})': p_i, f'P(k|{j})': p_j,
                     'ratio': p_i / p_j if p_j > 0 else np.nan})
    return pd.DataFrame(rows).set_index('k')

ratio_table('actor', 'director', ['performance', 'camera', 'the', 'car'])

**🤔 Quick check**
* `performance` is related to *actor* but not to *director* → large ratio
* `camera` is related to *director* → small ratio
* `the` is related to both → close to 1
* `car` is related to neither, and in 1.3 M tokens it simply never co-occurs: $0/0$.
  This is one reason GloVe is trained on billions of tokens, and why its loss must handle $X_{ij} = 0$.

GloVe learns vectors whose dot product reproduces these statistics, $w_i^\top \tilde{w}_k + b_i + \tilde{b}_k = \log X_{ik}$.

### &nbsp;3-3. The weighting function $f(X_{ij})$
The least-squares objective $J = \sum_{i,j} f(X_{ij}) \left(w_i^\top \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2$
weights every cell with

$$f(x) = \begin{cases} (x / x_{max})^{\alpha} & x < x_{max} \\ 1 & \text{otherwise} \end{cases}$$

which satisfies the three desiderata: $f(0) = 0$, non-decreasing, and not too large for very frequent pairs.

Since $\log 0$ is undefined, $f(0) = 0$ is read as "empty cells do not count": in practice the sum runs only over the
**non-zero** cells, $J = \sum_{X_{ij} > 0} f(X_{ij})(\dots)^2$.

In [ ]:
def glove_weight(x, x_max=100, alpha=0.75):
    x = np.asarray(x, dtype=float)
    return np.where(x < x_max, (x / x_max) ** alpha, 1.0)

x = np.linspace(0, 200, 400)
plt.figure(figsize=(6, 3))
plt.plot(x, glove_weight(x), color='seagreen')
plt.axvline(100, ls=':', color='slateblue')
plt.xlabel('$X_{ij}$'); plt.ylabel('$f(X_{ij})$'); plt.title('GloVe weighting, $x_{max}=100$, alpha = 3/4')
plt.show()

print('f(0), f(1), f(10), f(100), f(1000) =', glove_weight([0, 1, 10, 100, 1000]).round(3))

### &nbsp;3-4. Pre-trained GloVe vectors
Training GloVe needs a corpus far larger than our 2,000 reviews, so we load the **pre-trained** vectors published by
the authors (6 billion tokens of Wikipedia + Gigaword, 100 dimensions). This is how GloVe is used in practice.

In [ ]:
import gensim.downloader as api

# ~128 MB, about 1 minute on the first run. The file is cached for the rest of the session.
glove_vectors = api.load('glove-wiki-gigaword-100')

print('#vocabulary size:', len(glove_vectors))
print('#vector size    :', glove_vectors.vector_size)

In [ ]:
for w in ['actor', 'frog']:
    print(f"most similar to '{w}':", [x[0] for x in glove_vectors.most_similar(w, topn=7)])

In [ ]:
# The relationship pairs of the lecture's Results slide
tests = [('man', 'king', 'woman'),          # man : king = woman : ?
         ('france', 'paris', 'italy'),      # country : capital
         ('company', 'ceo', 'microsoft'),   # company : ceo
         ('strong', 'strongest', 'dark')]   # comparative - superlative
for a, b, c in tests:
    answer = glove_vectors.most_similar(positive=[b, c], negative=[a], topn=3)
    print(f'{a} : {b} = {c} : ?  ->', [w for w, _ in answer])

**🤔 Quick check**
`most_similar(positive=[b, c], negative=[a])` computes $b - a + c$ and, importantly, **never returns $a$, $b$ or $c$ themselves**.
Remove that rule and the nearest vector to *king - man + woman* is `king` again. The analogy works,
but less magically than it looks.

## &nbsp;4. FastText
*Limitation of NNLM, Word2Vec and GloVe*: each word gets a distinct vector, so morphology is ignored and
morphologically rich languages (Turkish, Finnish, Korean) with many rare forms suffer.

FastText represents a word as the **sum of the vectors of its character n-grams**:

* `<` and `>` mark the beginning and the end of the word
* all n-grams with $n$ = 3, 4, 5, 6 are kept
* the word itself, `<where>`, is also included as a special sequence (with its own vector, different from the n-gram `her` inside `where`)

### &nbsp;4-1. Character n-grams

In [ ]:
def char_ngrams(word, n_min=3, n_max=6):
    w = '<' + word + '>'
    grams = [w[i:i + n] for n in range(n_min, n_max + 1) for i in range(len(w) - n + 1)]
    if w not in grams:
        grams.append(w)           # the whole word as its own sequence
    return grams

for word in ['where', 'her']:
    g = char_ngrams(word)
    print(f'{word}: {len(g)} items ->', g)

`her` and `where` **share** the trigram `her`, so they share that n-gram vector. What they do not share is the whole-word
sequence: the word `her` also has its own vector `<her>`, which is a different string from the n-gram `her`
(the lecture: *different vectors are assigned to a word and an n-gram sharing the same sequence of characters*).

### &nbsp;4-2. Training FastText

In [ ]:
# Training takes about 1 minute.
ft_model = FastText(sentences=sentences, vector_size=100, window=5, min_count=5,
                    sg=1, workers=4, epochs=5, seed=0, min_n=3, max_n=6)
fasttext_vectors = ft_model.wv
print('#vocabulary size:', len(fasttext_vectors))

print("\nmost similar to 'actor' (FastText):", [x[0] for x in fasttext_vectors.most_similar('actor', topn=8)])

In [ ]:
# The OOV word that crashed Word2Vec now works
print("'hanyang' in vocabulary:", 'hanyang' in fasttext_vectors.key_to_index)
print(fasttext_vectors.most_similar('hanyang', topn=5))

# A misspelling and a new derivation
for w in ['actorr', 'unwatchable']:
    print(f"{w:<12} in vocab: {w in fasttext_vectors.key_to_index!s:<6} ->",
          [x[0] for x in fasttext_vectors.most_similar(w, topn=4)])

### &nbsp;4-3. A word vector built from its n-grams
For an OOV word gensim looks up the vector of every n-gram (hashed into `bucket` rows of `vectors_ngrams`) and
combines them. The lecture says *sum*; gensim takes the *mean*, which is the same direction, so cosine similarities do not change.

In [ ]:
from gensim.models.fasttext import ft_ngram_hashes

word = 'hanyang'
hashes = ft_ngram_hashes(word, ft_model.wv.min_n, ft_model.wv.max_n, ft_model.wv.bucket)
ngram_vecs = ft_model.wv.vectors_ngrams[hashes]

v_sum = ngram_vecs.sum(axis=0)
v_mean = ngram_vecs.mean(axis=0)
print('#n-grams used :', len(hashes))
print('mean == gensim:', np.allclose(v_mean, fasttext_vectors[word], atol=1e-6))   # atol: float32 rounding
print('cos(sum, mean):', round(float(v_sum @ v_mean / np.linalg.norm(v_sum) / np.linalg.norm(v_mean)), 4))

**❓ Why can FastText represent OOV words while Word2Vec cannot?**
An unseen word is still made of n-grams, and many of them were trained on other words (an n-gram never seen in training
only has a random hash-bucket vector). The neighbors of `hanyang` are words that *look* alike,
though: subword information captures form first, and meaning only as far as form carries it.

## &nbsp;5. Evaluating Embeddings
"The neighbors look reasonable" is not a measurement. Two standard intrinsic evaluations:

* **Word similarity test** : humans rated how similar 203 word pairs are; we correlate those ratings with the cosine similarity of the model.
* **Word analogy test** : `a : b = c : ?`, we check whether the expected word is among the model's top answers.

### &nbsp;5-1. Word similarity test

In [ ]:
import scipy.stats as st
import seaborn as sns

pairs = pd.read_csv(f'https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week06-text-representation-2/data/wordsim_similarity_goldstandard.csv', header=None, names=['w1', 'w2', 'human'])
print(pairs.shape)
pairs.head()

In [ ]:
def word_sim_test(pairs, wordvectors):
    actual, predicted, missed = [], [], 0
    for w1, w2, human in pairs.itertuples(index=False):
        try:
            predicted.append(wordvectors.similarity(w1, w2))   # cosine similarity of the model
            actual.append(float(human))                        # human judgement
        except KeyError:
            missed += 1                                        # a word is not in the vocabulary
    spearman, _ = st.spearmanr(actual, predicted)
    pearson, _ = st.pearsonr(actual, predicted)
    return spearman, pearson, missed


models = {'Word2Vec': word2vec_vectors, 'FastText': fasttext_vectors, 'GloVe (pre-trained)': glove_vectors}

rows = []
for name, wv in models.items():
    spearman, pearson, missed = word_sim_test(pairs, wv)
    print(f'{name:<22} spearman={spearman:.3f}  pearson={pearson:.3f}  missed pairs={missed}')
    rows += [{'model': name, 'criterion': 'spearman', 'score': spearman},
             {'model': name, 'criterion': 'pearson', 'score': pearson}]

plt.figure(figsize=(7, 3.5))
sns.barplot(x='model', y='score', hue='criterion', data=pd.DataFrame(rows))
plt.title('Word Similarity Test'); plt.ylim(0, 1); plt.show()

### &nbsp;5-2. Word analogy test
Lines look like `Athens Greece Baghdad Iraq`, read *Athens : Greece = Baghdad : ?*, so the query is
**Greece - Athens + Baghdad**. Lines starting with `:` are section names. Each question needs a `most_similar` call over the
whole vocabulary, so we evaluate on a 500-question sample. A question is answered only if **all four** words are in the
model's vocabulary (for FastText: in its word vocabulary, so the expected answer can actually be returned).

In [ ]:
questions = []
for line in pd.read_csv(f'https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week06-text-representation-2/data/word_analogy_goldstandard.txt', header=None, sep='\t')[0]:
    line = line.strip()
    if line.startswith(':') or len(line.split()) != 4:
        continue
    questions.append(line.lower().split())
print('#questions:', len(questions), ' example:', questions[0])


def word_analogy_test(questions, wordvectors, topn=10, sample=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(questions), size=min(sample, len(questions)), replace=False)
    correct, answered, missed = 0, 0, 0
    for i in idx:
        a, b, c, expected = questions[i]
        if not all(w in wordvectors.key_to_index for w in (a, b, c, expected)):
            missed += 1                                  # one of the four words is OOV: skip the question
            continue
        predicted = [w for w, _ in wordvectors.most_similar(positive=[b, c], negative=[a], topn=topn)]
        answered += 1
        correct += expected in predicted
    acc = correct / answered if answered else 0.0
    print(f'   answered={answered:>3}  skipped(OOV)={missed:>3}  accuracy(top-{topn})={acc:.3f}')
    return acc


for name, wv in models.items():
    print(name)
    word_analogy_test(questions, wv)

**🤔 Quick check** Read the accuracy together with `answered`. Word2Vec and FastText share the movie-review vocabulary,
so they answer the **same** small set of questions, and on that same set FastText is far more accurate: many questions are
grammatical (`amazing : amazingly`, `walk : walking`), exactly what character n-grams capture. GloVe, trained on
6 billion tokens, answers all of them. The size and coverage of the training corpus matter as much as the algorithm.

---
### ✅ What we practiced
* One-hot vectors cannot express similarity; a distributed representation is a learned map $W$
* Word2Vec : the skip-gram training pairs, subsampling, Skip-gram vs CBOW, and the two vectors $v$ (kept) and $u$ of every word
* GloVe : $X_{ij}$, $X_i$ as a row sum, the probability ratio, the weighting function $f(x)$, pre-trained vectors
* FastText : character n-grams with `<` `>` and the whole word, vectors for unseen words
* Intrinsic evaluation : word similarity and word analogy

**Next: Notebook 2** goes from words to sentences and documents (chapter 4, Doc2Vec).
Visualizing these vectors in 2-D (t-SNE) comes in Week 7, after the dimensionality reduction lecture.